In [ ]:
from pathlib import Path
import sys

# Locate this package and load the shared v3 LE locations (config/paths.json,
# overridable with the V3LE_ROOT environment variable).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from paths import V3LE_DATA_DIR, V3LE_DIAG_DIR, V3LE_FIG_ROOT, fig_dir, require

In [39]:
import os
import glob
import re
import json
import time
import datetime
import numpy as np
import pandas as pd
import shutil
import psutil
import subprocess

import xarray as xr
import xcdat as xc
from xcdat import open_dataset as xcdat_open
from pathlib import Path


In [40]:
def read_exp_dict():    
    run_dict = {
        'NOAAv5': {'name': 'NOAAv5', 'period': '185001-202512'},
    }
    return run_dict 

In [41]:
def read_var_dict():
    var_dict = {
            'TS' : {'alias': 'anom', 'unit': 'K',            'min': -6,  'max': 6,  'nlev': 11, },
        }
    return var_dict 

In [44]:
class AnnualAnomalyProcessor:
    """
    Compute annual means and anomalies (relative to a reference year) for a set of variables,
    optionally with an ice/land mask, and save one NetCDF per experiment.
    """
    def __init__(
        self,
        data_dir,
        group,
        run_dict,
        var_dict,
        results_dir,
        region="global",
        reference_year=1850,
        frequency="monthly",
        lon_name="lon",
        mask_kind="ice",          # "ice", "land", or None
        mask_var="SICONC",
        mask_threshold=15.0,      # % for ice, >0 for land by default
        min_sw=1.0                # small guard for SW albedo divisions
    ):
        self.data_dir = data_dir
        self.group = group
        self.run_dict = run_dict
        self.var_dict = var_dict
        self.results_dir = results_dir
        self.region = region
        self.reference_year = reference_year
        self.frequency = frequency
        self.lon_name = lon_name
        self.mask_kind = mask_kind
        self.mask_var = mask_var
        self.mask_threshold = mask_threshold
        self.min_sw = float(min_sw)

        self.region_all = self.get_default_region()
        
        if region not in self.region_all:
            raise ValueError(f"Unknown region '{region}'. Choose from {list(self.REGIONS)}.")
        os.makedirs(self.results_dir, exist_ok=True)

    # ---------- Public API ----------
    def get_default_region(self):
            # ---------- Configuration ----------
        region_dict = {
            "global":   [(-90, 90), (-180, 180)],
            "Atlantic": [(0, 60), (265, 345)],
            "Arctic":   [(50, 90), (-180, 180)],
            "CONUS":    [(25, 50), (235, 295)],
            "Antarctic":[(-90, -50), (-180, 180)],
            "PolarN":   [(50, 90), (-180, 180)],
            "Greenland":[(55, 90), (-75, -10)],
        }
        return region_dict
    
    def process_all(self):
        for exp in self.run_dict:
            self.process_one(exp)

    def process_one(self, exp):
        period = self.run_dict[exp]["period"]
        lat_slice, lon_slice = self._region_slices(self.region)

        # 1) Load all raw + expr variables
        ds_all = self._load_all_vars_for_run(exp, period, lat_slice, lon_slice)

        # 2) Compute derived fields (CRE, albedo, etc.)
        self._compute_derived_fields(ds_all)

        # 3) Build mask once per run
        mask = self._build_mask(exp, period, lat_slice, lon_slice)

        # 4) Annual mean and anomalies (masked & unmasked)
        out = self._annual_mean_and_anomaly(ds_all, mask)

        # 5) Save
        out_path = os.path.join(self.results_dir, f"{exp}.annual_mean_anomaly.{period}.nc")
        out.to_netcdf(out_path)
        print(f"[INFO] Saved annual mean + anomaly to: {out_path}")

    # ---------- Helpers: region, lon, time ----------
    def _region_slices(self, regnam):
        (lat0, lat1), (lon0, lon1) = self.region_all[regnam]
        return slice(lat0, lat1), slice(lon0, lon1)

    @staticmethod
    def _normalize_longitude(ds, lon_name="lon"):
        lons = ds[lon_name].values
        if np.min(lons) > -1:
            ds = ds.assign_coords({lon_name: ((lons + 180) % 360 - 180)})
            ds = ds.sortby(lon_name)
        return ds

    @staticmethod
    def _ensure_valid_time(ds, period, frequency="monthly"):
        """
        Reconstructs monthly time coordinate if missing or undecodable, using 'YYYYMM-YYYYMM'.
        """
        try:
            t = ds["time"]
            if hasattr(t, "dt") and np.issubdtype(t.dtype, np.datetime64):
                return ds
        except Exception:
            pass

        start_str, end_str = period.split("-")
        start = pd.to_datetime(f"{start_str[:4]}-{start_str[4:]}-01")
        end = pd.to_datetime(f"{end_str[:4]}-{end_str[4:]}-01")
        new_time = pd.date_range(start=start, end=end, freq="MS")

        nt = ds.sizes.get("time", len(new_time))
        if nt != len(new_time):
            raise ValueError(
                f"[ERROR] Expected {len(new_time)} monthly steps from {period}, but dataset has {nt}."
            )
        print(f"[INFO] Reconstructing time from {start.date()} to {end.date()}")
        return ds.assign_coords(time=new_time)

    # ---------- IO ----------
    def _open_subset(self, path, period, lat_slice, lon_slice, vin=None):
        ds = xcdat_open(path, decode_times=True)
        ds = self._ensure_valid_time(ds, period, self.frequency)
        ds = self._normalize_longitude(ds, self.lon_name)
        ds = ds.bounds.add_missing_bounds()
        return ds.sel(lat=lat_slice, lon=lon_slice)
    
    # Helper to open a subset
    def _open(self, exp, vnam, period, lat_slice, lon_slice, vin=None):
        path = os.path.join(self.data_dir, self.group, f"{exp}.{vnam}.{period}.nc")
        ds = self._open_subset(path, period, lat_slice, lon_slice)
        print(vin)
        print(xxxxx)
        if vin is not None:
            dx = ds[vin]
        else:
            dx = ds[vnam]
        return dx
        
    def _open_var(self, exp, var, period, lat_slice, lon_slice, vin = None):
        """
        Open a variable with smart fallbacks + unit harmonization.

        - If var == "SICONC" but file missing, try ICEFRAC (fraction) → convert to percent → rename to SICONC.
        - If var == "SST"    but file missing, try TS (K) → convert to degC → mask to ocean → rename to SST.
        - Otherwise open the requested file directly.

        Returns
        -------
        xr.DataArray
        """

        # --- primary attempt ---
        path = os.path.join(self.data_dir, self.group, f"{exp}.{var}.{period}.nc")
        if os.path.exists(path):
            if vin is not None:
                da = self._open_subset(path, period, lat_slice, lon_slice)[vin]
            else:
                da = self._open_subset(path, period, lat_slice, lon_slice)[var]
            return da  # already in requested name/units

        # --- fallbacks ---
        # 1) SICONC <- ICEFRAC (fraction → percent)
        if var == "SICONC":
            alt_var = "ICEFRAC"
            alt_path = os.path.join(self.data_dir, self.group, f"{exp}.{alt_var}.{period}.nc")
            if os.path.exists(alt_path):
                ice = self._open(exp, alt_var, period, lat_slice, lon_slice, vin = vin)
                # convert fraction to percent
                out = (ice * 100.0).rename("SICONC")
                # update attrs
                out.attrs = dict(ice.attrs)
                out.attrs["units"] = "%"
                out.attrs["long_name"] = out.attrs.get("long_name", "Sea Ice Concentration")
                return out
            raise FileNotFoundError(f"Neither SICONC nor ICEFRAC file found for {exp} {period}")

        # 2) SST <- TS (K → °C) + mask to ocean
        if var == "SST":
            alt_var = "TS"
            alt_path = os.path.join(self.data_dir, self.group, f"{exp}.{alt_var}.{period}.nc")
            if os.path.exists(alt_path):
                ts = self._open(exp, alt_var, period, lat_slice, lon_slice,  vin = vin)
                # K -> °C
                sst = (ts - 273.15).rename("SST")
                sst.attrs = dict(ts.attrs)
                sst.attrs["units"] = "degC"
                sst.attrs["long_name"] = sst.attrs.get("long_name", "Sea Surface Temperature (derived from TS)")

                # mask to ocean (fraction > threshold)
                ocean_mask = self._get_ocean_mask(exp, period, lat_slice, lon_slice)
                sst = sst.where(ocean_mask)

                return sst
            raise FileNotFoundError(f"Neither SST nor TS file found for {exp} {period}")

        # If we reach here, there was no special fallback rule for `var`
        raise FileNotFoundError(f"File for {var} not found: {path}")


    def _get_ocean_mask(self, exp, period, lat_slice, lon_slice):
        """
        Return a boolean mask (True = ocean) using available fraction fields.
        Tries, in order:
          - OCNFRAC (0..1: ocean fraction)
          - LANDFRAC (0..1: land fraction)
          - SFTLF   (0..100: percent land fraction)
        Uses threshold self.ocean_frac_thresh (default 0.5).
        """
        def _try(vnam):
            pth = os.path.join(self.data_dir, self.group, f"{exp}.{vnam}.{period}.nc")
            if os.path.exists(pth):
                return self._open_subset(pth, period, lat_slice, lon_slice)[vnam]
            return None

        thr = getattr(self, "ocean_frac_thresh", 0.5)

        ocnfrac = _try("OCNFRAC")
        if ocnfrac is not None:
            return (ocnfrac > thr)

        landfrac = _try("LANDFRAC")
        if landfrac is not None:
            return (1.0 - landfrac) > thr

        sftlf = _try("SFTLF")  # percent land
        if sftlf is not None:
            ocnfrac_guess = 1.0 - (sftlf / 100.0)
            return (ocnfrac_guess > thr)

        # As a last resort, allow everything (no mask) or raise; raising is safer.
        raise FileNotFoundError(
            "Cannot construct ocean mask: missing OCNFRAC, LANDFRAC, and SFTLF."
        )

    # ---------- Variable loading ----------
    def _eval_expr(self, exp, expr, period, lat_slice, lon_slice, vin=None):
        """Safely evaluate a simple algebraic expression of already-available variables."""
        expr = expr.replace("−", "-")
        tokens = [v.strip() for v in expr.replace("+", " ").replace("-", " ").split()]
        scope = {}
        for v in tokens:
            if v and v not in scope:
                scope[v] = self._open_var(exp, v, period, lat_slice, lon_slice, vin = vin)
        # Disable builtins, use only our variables:
        return eval(expr, {"__builtins__": {}}, scope)

    def _load_all_vars_for_run(self, exp, period, lat_slice, lon_slice):
        ds_all = {}
        for vname, meta in self.var_dict.items():
            vexpr = meta.get("expr")
            vin = meta.get("alias")
            if vexpr is None:
                da = self._open_var(exp, vname, period, lat_slice, lon_slice, vin = vin)
            else:
                da = self._eval_expr(exp, vexpr, period, lat_slice, lon_slice, vin = vin)
                
            # Debug time range
            tvals = da["time"].values
            print(f"[DEBUG] {exp}:{vname} time range = {str(tvals[0])} → {str(tvals[-1])}")
        return ds_all

    # ---------- Derived fields ----------
    def _compute_derived_fields(self, ds_all):
        sigma = 5.67e-8

        # Upwelling LW (clear) from TS
        if "TS" in ds_all and "FLUSC" not in ds_all:
            ds_all["FLUSC"] = sigma * ds_all["TS"] ** 4
            ds_all["FLUSC"].attrs.update({"long_name": "Surface upwelling LW (clear-sky)", "units": "W m-2"})

        # Downwelling LW (clear) if missing
        if "FLNSC" in ds_all and "FLUSC" in ds_all and "FLDSC" not in ds_all:
            ds_all["FLDSC"] = ds_all["FLNSC"] + ds_all["FLUSC"]
            ds_all["FLDSC"].attrs.update({"long_name": "Surface downwelling LW (clear-sky)", "units": "W m-2"})

        # Surface CRE components (all-sky vs clear-sky)
        if "FSNS" in ds_all and "FSNSC" in ds_all:
            ds_all["SWCF_SRF"] = ds_all["FSNS"] - ds_all["FSNSC"]
            ds_all["SWCF_SRF"].attrs.update({"long_name": "Surface SW cloud radiative effect", "units": "W m-2"})

        if "FLNS" in ds_all and "FLNSC" in ds_all:
            ds_all["LWCF_SRF"] = ds_all["FLNS"] - ds_all["FLNSC"]
            ds_all["LWCF_SRF"].attrs.update({"long_name": "Surface LW cloud radiative effect", "units": "W m-2"})

        if "SWCF_SRF" in ds_all and "LWCF_SRF" in ds_all:
            ds_all["CRE_SRF"] = ds_all["SWCF_SRF"] + ds_all["LWCF_SRF"]
            ds_all["CRE_SRF"].attrs.update({"long_name": "Surface cloud radiative effect", "units": "W m-2"})

        # TOA CRE
        if "FSNT" in ds_all and "FSNTC" in ds_all:
            ds_all["SWCF"] = ds_all["FSNT"]  - ds_all["FSNTC"]
            ds_all["SWCF"].attrs.update({"long_name": "TOA SW cloud radiative effect", "units": "W m-2"})

        if "FLNT" in ds_all and "FLNTC" in ds_all:
            ds_all["LWCF"] = ds_all["FLNTC"] - ds_all["FLNT"]
            ds_all["LWCF"].attrs.update({"long_name": "TOA LW cloud radiative effect", "units": "W m-2"})

        if "SWCF" in ds_all and "LWCF" in ds_all:
            ds_all["CRE"] = ds_all["SWCF"] + ds_all["LWCF"]
            ds_all["CRE"].attrs.update({"long_name": "TOA cloud radiative effect", "units": "W m-2"})

        # Surface albedo (all-sky & clear-sky). Also expose FSUS/FSUSC if missing.
        if "FSDS" in ds_all and "FSNS" in ds_all:
            ds_all.setdefault("FSUS", ds_all["FSDS"] - ds_all["FSNS"])
            alb = 1.0 - (ds_all["FSNS"] / ds_all["FSDS"]).where(ds_all["FSDS"] > self.min_sw)
            ds_all["ALBEDO_SRF"] = alb.clip(0.0, 1.0)
            ds_all["ALBEDO_SRF"].attrs.update({"long_name": "Surface albedo (all-sky)", "units": "1"})

        if "FSDSC" in ds_all and "FSNSC" in ds_all:
            ds_all.setdefault("FSUSC", ds_all["FSDSC"] - ds_all["FSNSC"])
            alb_c = 1.0 - (ds_all["FSNSC"] / ds_all["FSDSC"]).where(ds_all["FSDSC"] > self.min_sw)
            ds_all["ALBEDOC_SRF"] = alb_c.clip(0.0, 1.0)
            ds_all["ALBEDOC_SRF"].attrs.update({"long_name": "Surface albedo (clear-sky)", "units": "1"})

    # ---------- Masks ----------
    def _build_mask(self, exp, period, lat_slice, lon_slice):
        if self.mask_kind is None:
            return None

        # Simple land/ice mask from a single variable threshold
        var = self.mask_var
        da = self._open_var(exp, var, period, lat_slice, lon_slice)

        if self.mask_kind == "ice":
            # e.g., SICONC > 15%
            mask = xr.where(da > self.mask_threshold, 1, 0)
        elif self.mask_kind == "land":
            # generic: > threshold ⇒ land (customize to LANDFRAC if available)
            mask = xr.where(da > self.mask_threshold, 1, 0)
        else:
            raise ValueError("mask_kind must be 'ice', 'land', or None.")
        return mask

    # ---------- Annual & anomaly ----------
    def _annual_mean_and_anomaly(self, ds_all, mask):
        out = xr.Dataset()
        ref_year = self.reference_year
        
        # Make sure ref_year is an int
        ref_year = int(ref_year)
        
        for vname, da in ds_all.items():
            # annual mean
            ann = da.groupby("time.year").mean("time")
            #print(ann["year"])
            if ref_year not in ann["year"]:
                raise ValueError(f"[ERROR] Reference year {ref_year} not in data for '{vname}'")

            ref = ann.sel(year=ref_year)
            out[vname] = ann
            out[f"D{vname}"] = ann - ref

            # masked version
            if mask is not None:
                if mask.ndim == 2:
                    da_masked = da.where(mask)
                elif mask.ndim == 3 and "time" in da.dims:
                    da_masked = da.where(mask)
                else:
                    raise ValueError("[ERROR] mask shape must be 2D(lat,lon) or 3D(time,lat,lon) aligned with data")

                ann_m = da_masked.groupby("time.year").mean("time")
                ref_m = ann_m.sel(year=ref_year)
                out[f"{vname}_imsk"] = ann_m
                out[f"D{vname}_imsk"] = ann_m - ref_m

        return out

In [45]:
if __name__ == "__main__":
    # --- Paths
    data_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/data"
    out_dir  = f"{V3LE_DATA_DIR}"
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    
    mask_kind=None
    mask_var=None
    mask_threshold=15.0 # only keep grid point with SICONC > 15%
    new_period = "185001-202508" 

    regnam = "global"
    group = 'NOAAv5'    
    ref_year = "1981" 
    
    filename_tag   = "monthly_base"
    
    var_dict = read_var_dict()
    run_dict = read_exp_dict()
    # unify all periods
    for key in run_dict:
        run_dict[key]["period"] = new_period

    results_dir = os.path.join(out_dir, regnam)
    Path(results_dir).mkdir(parents=True, exist_ok=True)

    # --- Monthly-only processor
    processor = MonthlyBaseProcessor(
        data_dir=data_dir,
        group=group,
        run_dict=run_dict,
        var_dict=var_dict,
        results_dir=results_dir,
        region=regnam,
        reference_year=ref_year,   # stored in attrs only (no seasonal/annual here)
        mask_kind=mask_kind,
        mask_var=mask_var,
        mask_threshold=mask_threshold,
    )

    # Build & save monthly base datasets for all experiments
    # (no seasonal/annual aggregation here)
    processor.process_all_monthly_bases(
        apply_mask=True,           # also writes *_imsk variables; set False to skip
        filename_tag=filename_tag,
        compress=True              # NetCDF compression + time chunking
    )
    

[INFO] Reconstructing time from 1850-01-01 to 2025-08-01


KeyError: "No variable named 'TS'. Variables on the dataset include ['lat', 'z', 'anom', 'time', 'lon']"